# LongCat-Video (G4)

Single-GPU BF16 text-to-video, image-to-video and video continuation; models on local SSD, generated videos on Drive, Caddy authentication.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "longcat-video")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "7860"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not os.path.isdir("/content/drive/MyDrive"):                 # Drive not mounted -> local fallback
    DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    logf = open(f"{WORK}/{name}.log", "a")
    return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        os.chmod(path, os.stat(path).st_mode | stat.S_IEXEC)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- LongCat-Video on local SSD, Blackwell-compatible CUDA environment
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

LC_WORK = Path("/content/longcat-video")
LC_REPO = LC_WORK / "repo"
LC_MODEL = LC_WORK / "model"
LC_VENV = LC_WORK / "venv"
LC_REVISION = "6b3f4b8582a8bc3f20f795735f5383716c4ba794"
LC_WORK.mkdir(parents=True, exist_ok=True)
LC_OUTPUT = Path(DATA_DIR) / "output"
LC_OUTPUT.mkdir(parents=True, exist_ok=True)
LC_ENV = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0",
          "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}
LC_CACHE_PAIRS = []
for name in ("triton-cache", "inductor-cache"):
    local_cache, drive_cache = LC_WORK / name, Path(DATA_DIR) / name
    local_cache.mkdir(exist_ok=True)
    drive_cache.mkdir(exist_ok=True)
    subprocess.run(["cp", "-rn", f"{drive_cache}/.", f"{local_cache}/"], check=False)
    LC_CACHE_PAIRS.append((local_cache, drive_cache))

if not (LC_REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/meituan-longcat/LongCat-Video.git", str(LC_REPO)], check=True)
subprocess.run(["git", "-C", str(LC_REPO), "checkout", "--detach", LC_REVISION], check=True)

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if not (LC_VENV / "bin/python").exists():
    subprocess.run([UV, "venv", "--python", "3.11", "--seed", str(LC_VENV)], check=True)
LC_PYTHON = str(LC_VENV / "bin/python")

def lc_install():
    marker = LC_WORK / "environment-v1.ok"
    if marker.exists():
        return
    log_path = LC_WORK / "install.log"
    with log_path.open("a") as log_file:
        def run(args):
            subprocess.run(args, stdout=log_file, stderr=subprocess.STDOUT, env=LC_ENV, check=True)
        # Upstream's torch 2.6/cu124 predates sm_120; use CUDA 13 wheels for G4.
        run([UV, "pip", "install", "--python", LC_PYTHON, "torch==2.11.0", "torchvision==0.26.0",
             "--index-url", "https://download.pytorch.org/whl/cu130"])
        run([UV, "pip", "install", "--python", LC_PYTHON,
             "numpy==1.26.4", "transformers==4.41.0", "diffusers==0.35.1",
             "einops==0.8.0", "ftfy==6.2.0", "loguru==0.7.2", "psutil",
             "av==12.0.0", "opencv-python-headless==4.9.0.80", "imageio==2.37.0",
             "imageio-ffmpeg==0.6.0", "huggingface-hub==0.36.0", "hf_xet",
             "accelerate>=1,<2", "sentencepiece", "gradio==5.49.1", "uvicorn", "fastapi"])
    marker.write_text("ok\n")

def lc_download():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface-hub", "hf_xet"], check=True)
    # snapshot_download validates the local snapshot on each run; interrupted downloads resume.
    download = (
        "from huggingface_hub import snapshot_download; "
        "snapshot_download('meituan-longcat/LongCat-Video', local_dir=" + repr(str(LC_MODEL)) +
        ", allow_patterns=['*.json', 'dit/*', 'text_encoder/*', 'tokenizer/*', 'vae/*', "
        "'scheduler/*', 'lora/cfg_step_lora.safetensors'])"
    )
    with (LC_WORK / "download.log").open("a") as log_file:
        subprocess.run([sys.executable, "-c", download], stdout=log_file, stderr=subprocess.STDOUT,
                       env=LC_ENV, check=True)
    shutil.rmtree("/root/.cache/huggingface/xet", ignore_errors=True)

with ThreadPoolExecutor(max_workers=2) as pool:
    jobs = [pool.submit(lc_install), pool.submit(lc_download)]
    while not all(job.done() for job in jobs):
        for job in jobs:
            if job.done():
                job.result()
        print("LongCat setup: environment=" + ("ready" if jobs[0].done() else "installing") +
              " model=" + ("ready" if jobs[1].done() else "downloading") +
              f" SSD free={shutil.disk_usage('/content').free / 1e9:.1f} GB; logs={LC_WORK}", flush=True)
        time.sleep(20)
    for job in jobs:
        job.result()
print("LongCat environment and model ready", flush=True)


In [ ]:
# @title 2. Service -- load BF16 model, verify inference, then serve Gradio on loopback
LC_APP = LC_WORK / "longcat_video_app.py"
LC_APP.write_text('"""Single-GPU LongCat-Video UI; embedded into the self-contained Colab notebook."""\nimport json\nimport os\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.nn.attention import SDPBackend, sdpa_kernel\n\n\ndef install_sdpa():\n    """Use native fused SDPA instead of the upstream CUDA-12.4 FlashAttention build.\n\n    Cross attention uses packed, variable-length text; split each sample before\n    calling SDPA so padding and neighbouring prompts never enter its attention.\n    This adapter is intentionally single GPU, matching the Colab deployment.\n    """\n    from longcat_video.modules.attention import Attention, MultiHeadCrossAttention\n\n    def self_attention(self, q, k, v, shape):\n        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):\n            return F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, scale=self.scale)\n\n    def cross_attention(self, x, cond, kv_seqlen):\n        batch, tokens, channels = x.shape\n        q = self.q_norm(self.q_linear(x).view(1, -1, self.num_heads, self.head_dim))\n        kv = self.kv_linear(cond).view(1, -1, 2, self.num_heads, self.head_dim)\n        k, v = kv.unbind(2)\n        k = self.k_norm(k)\n        outputs, offset = [], 0\n        for i, length in enumerate(kv_seqlen):\n            qi = q[:, i * tokens:(i + 1) * tokens].transpose(1, 2)\n            ki = k[:, offset:offset + length].transpose(1, 2)\n            vi = v[:, offset:offset + length].transpose(1, 2)\n            with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):\n                out = F.scaled_dot_product_attention(qi, ki, vi, dropout_p=0.0)\n            outputs.append(out.transpose(1, 2))\n            offset += length\n        return self.proj(torch.cat(outputs, dim=1).reshape(batch, tokens, channels))\n\n    Attention._process_attn = self_attention\n    MultiHeadCrossAttention._process_cross_attn = cross_attention\n\n\ndef load_pipeline():\n    from transformers import AutoTokenizer, UMT5EncoderModel\n    from longcat_video.pipeline_longcat_video import LongCatVideoPipeline\n    from longcat_video.modules.scheduling_flow_match_euler_discrete import FlowMatchEulerDiscreteScheduler\n    from longcat_video.modules.autoencoder_kl_wan import AutoencoderKLWan\n    from longcat_video.modules.longcat_video_dit import LongCatVideoTransformer3DModel\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("LongCat-Video requires a CUDA GPU")\n    install_sdpa()\n    model = os.environ["CT_LONGCAT_MODEL_DIR"]\n    common = dict(torch_dtype=torch.bfloat16, local_files_only=True)\n    tokenizer = AutoTokenizer.from_pretrained(model, subfolder="tokenizer", local_files_only=True)\n    text_encoder = UMT5EncoderModel.from_pretrained(model, subfolder="text_encoder", **common)\n    vae = AutoencoderKLWan.from_pretrained(model, subfolder="vae", **common)\n    dit = LongCatVideoTransformer3DModel.from_pretrained(\n        model, subfolder="dit", cp_split_hw=[1, 1],\n        enable_flashattn2=False, enable_flashattn3=False, enable_xformers=False,\n        **common,\n    )\n    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(model, subfolder="scheduler")\n    for module in (text_encoder, vae, dit):\n        module.eval().requires_grad_(False)\n    pipe = LongCatVideoPipeline(tokenizer, text_encoder, vae, scheduler, dit)\n    pipe.dit.load_lora(str(Path(model) / "lora/cfg_step_lora.safetensors"), "cfg_step_lora")\n    pipe.to("cuda")\n    return pipe\n\n\ndef main():\n    import gradio as gr\n    import uvicorn\n    from fastapi import FastAPI\n    from diffusers.utils import export_to_video, load_image, load_video\n\n    output_dir = Path(os.environ["CT_LONGCAT_OUTPUT_DIR"])\n    output_dir.mkdir(parents=True, exist_ok=True)\n    pipe = load_pipeline()\n    lock = threading.Lock()\n    # A real inference pass is the readiness gate, not just an HTTP listener.\n    with torch.inference_mode():\n        smoke = pipe.generate_t2v(\n            prompt="A small orange cat walking through a sunny garden.",\n            height=256, width=256, num_frames=9, num_inference_steps=1,\n            guidance_scale=1.0, generator=torch.Generator(device="cuda").manual_seed(42),\n        )[0]\n    smoke_path = Path(os.environ["CT_LONGCAT_WORK_DIR"]) / "smoke.mp4"\n    export_to_video(smoke, str(smoke_path), fps=15)\n    print(f"LONGCAT_INFERENCE_OK frames={len(smoke)} output={smoke_path}", flush=True)\n    del smoke\n    torch.cuda.empty_cache()\n\n    def generate(task, prompt, negative, image, video, resolution, width, height,\n                 frames, steps, guidance, distill, seed, segments):\n        if not prompt.strip():\n            raise gr.Error("请输入视频描述")\n        if task == "图生视频" and not image:\n            raise gr.Error("请上传参考图片")\n        if task == "视频续写" and not video:\n            raise gr.Error("请上传参考视频")\n        width, height, frames = int(width), int(height), int(frames)\n        if not (256 <= width <= 1024 and 256 <= height <= 1024 and 17 <= frames <= 161):\n            raise gr.Error("宽高范围为 256–1024，帧数范围为 17–161")\n        if width % 16 or height % 16 or frames % 4 != 1:\n            raise gr.Error("宽高须为 16 的倍数，帧数须为 4n+1")\n        if task not in ("文生视频", "图生视频", "视频续写") or resolution not in ("480p", "720p"):\n            raise gr.Error("请选择有效的生成模式与分辨率")\n        if not (1 <= int(segments) <= 10 and 1 <= int(steps) <= 100 and 1 <= float(guidance) <= 10):\n            raise gr.Error("生成段数、步数或引导强度超出范围")\n        seed = int(seed)\n        if seed < 0:\n            seed = int.from_bytes(os.urandom(4), "little")\n        started = time.time()\n        with lock, torch.inference_mode():\n            try:\n                if distill:\n                    pipe.dit.enable_loras(["cfg_step_lora"])\n                generator = torch.Generator(device="cuda").manual_seed(seed)\n                kwargs = dict(prompt=prompt, negative_prompt=None if distill else negative,\n                              num_frames=frames, num_inference_steps=16 if distill else int(steps),\n                              guidance_scale=1.0 if distill else float(guidance),\n                              use_distill=distill, generator=generator)\n                if task == "文生视频":\n                    if int(segments) > 1:\n                        from PIL import Image\n                        # Continuation uses upstream aspect buckets: select the same\n                        # bucket for the first segment so every clip can be joined.\n                        height, width = pipe.get_condition_shape(\n                            Image.new("RGB", (width, height)), resolution,\n                            scale_factor_spatial=pipe.vae_scale_factor_spatial * 2,\n                        )\n                    result = pipe.generate_t2v(height=height, width=width, **kwargs)[0]\n                elif task == "图生视频":\n                    result = pipe.generate_i2v(image=load_image(image), resolution=resolution, **kwargs)[0]\n                else:\n                    import cv2\n                    cap = cv2.VideoCapture(video)\n                    fps = cap.get(cv2.CAP_PROP_FPS) or 15\n                    cap.release()\n                    source = load_video(video)[::max(1, round(fps / 15))]\n                    if len(source) < 13:\n                        raise gr.Error("参考视频须至少包含 13 帧（按 15 fps 采样）")\n                    result = pipe.generate_vc(video=source, resolution=resolution, num_cond_frames=13,\n                                              use_kv_cache=True, offload_kv_cache=True,\n                                              enhance_hf=not distill, **kwargs)[0]\n                clips = [result]\n                for _ in range(1, int(segments)):\n                    from PIL import Image\n                    source = [Image.fromarray((frame * 255).clip(0, 255).astype(np.uint8)) for frame in result[-13:]]\n                    result = pipe.generate_vc(video=source, resolution=resolution, num_cond_frames=13,\n                                              use_kv_cache=True, offload_kv_cache=True,\n                                              enhance_hf=not distill, **kwargs)[0]\n                    clips.append(result[13:])\n                result = np.concatenate(clips, axis=0)\n                name = f"{time.strftime(\'%Y%m%d-%H%M%S\')}-{uuid.uuid4().hex[:8]}"\n                path = output_dir / f"{name}.mp4"\n                export_to_video(result, str(path), fps=15)\n                metadata = dict(task=task, prompt=prompt, negative_prompt=negative, seed=seed,\n                                frames=len(result), fps=15, resolution=resolution,\n                                width=int(result.shape[2]), height=int(result.shape[1]),\n                                steps=16 if distill else int(steps), guidance=1.0 if distill else guidance,\n                                distill=distill, segments=int(segments), elapsed_seconds=round(time.time()-started, 1))\n                path.with_suffix(\'.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))\n                return str(path), str(path), metadata\n            finally:\n                pipe.dit.disable_all_loras()\n                torch.cuda.empty_cache()\n\n    with gr.Blocks(title="LongCat-Video") as demo:\n        gr.Markdown("# LongCat-Video\\n文生视频 · 图生视频 · 视频续写。生成结果自动保存到 Google Drive。")\n        with gr.Row():\n            with gr.Column():\n                task = gr.Dropdown(["文生视频", "图生视频", "视频续写"], value="文生视频", label="生成模式")\n                prompt = gr.Textbox(label="视频描述", lines=4, placeholder="描述画面、人物动作和镜头运动，建议使用英文")\n                negative = gr.Textbox(label="负面描述", value="blurry, low quality, static, subtitles")\n                image = gr.Image(type="filepath", label="参考图片（图生视频）")\n                video = gr.Video(label="参考视频（视频续写）")\n                distill = gr.Checkbox(value=True, label="快速模式（16 步）")\n                with gr.Accordion("生成参数", open=False):\n                    resolution = gr.Dropdown(["480p", "720p"], value="480p", label="图片 / 视频续写 / 多段生成分辨率")\n                    width = gr.Slider(256, 1024, value=832, step=16, label="文生视频宽度")\n                    height = gr.Slider(256, 1024, value=480, step=16, label="文生视频高度")\n                    frames = gr.Slider(17, 161, value=93, step=4, label="每段帧数（15 fps）")\n                    steps = gr.Slider(1, 100, value=50, step=1, label="标准模式采样步数")\n                    guidance = gr.Slider(1, 10, value=4, step=0.5, label="标准模式引导强度")\n                    seed = gr.Number(value=42, precision=0, label="随机种子（-1 为随机）")\n                    segments = gr.Slider(1, 10, value=1, step=1, label="连续生成段数")\n                submit = gr.Button("生成视频", variant="primary")\n            with gr.Column():\n                result = gr.Video(label="生成结果")\n                download = gr.File(label="下载视频")\n                metadata = gr.JSON(label="生成信息")\n        submit.click(generate, [task, prompt, negative, image, video, resolution, width, height,\n                               frames, steps, guidance, distill, seed, segments],\n                     [result, download, metadata], api_name="generate", concurrency_limit=1)\n    api = FastAPI()\n\n    @api.get(\'/health\')\n    def health():\n        return {"status": "ok", "model": "LongCat-Video", "gpu": torch.cuda.get_device_name(0),\n                "inference_verified": True}\n\n    demo.queue(max_size=8, default_concurrency_limit=1)\n    public_host = os.environ.get("CT_PUBLIC_HOSTNAME", "")\n    api = gr.mount_gradio_app(\n        api, demo, path="/", allowed_paths=[str(output_dir)],\n        root_path=f"https://{public_host}" if public_host else None,\n    )\n    uvicorn.run(api, host="127.0.0.1", port=int(os.environ.get("CT_SERVICE_PORT", "7860")))\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")

def start_service():
    return ct_spawn("service", [LC_PYTHON, str(LC_APP)], cwd=str(LC_REPO), env={
        "PYTHONPATH": str(LC_REPO), "PYTHONUNBUFFERED": "1",
        "CT_LONGCAT_MODEL_DIR": str(LC_MODEL), "CT_LONGCAT_WORK_DIR": str(LC_WORK),
        "CT_LONGCAT_OUTPUT_DIR": str(LC_OUTPUT),
        "CT_SERVICE_PORT": str(SERVICE_PORT),
        "TRITON_CACHE_DIR": str(LC_WORK / "triton-cache"),
        "TORCHINDUCTOR_CACHE_DIR": str(LC_WORK / "inductor-cache"),
        "GRADIO_ANALYTICS_ENABLED": "False",
    })

PROCS["service"] = (start_service(), start_service)
# Wait for model + real inference before the shared authentication/tunnel cells.
deadline = time.time() + 1800
while time.time() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("LongCat service exited:\n" + ct_tail("service", 60))
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=3) as response:
            if json.load(response).get("inference_verified"):
                break
    except (OSError, ValueError):
        pass
    print("LongCat loading / warming up:\n" + ct_tail("service", 3), flush=True)
    time.sleep(20)
else:
    raise TimeoutError("LongCat inference readiness timed out:\n" + ct_tail("service", 60))
for local_cache, drive_cache in LC_CACHE_PAIRS:
    subprocess.run(["cp", "-ru", f"{local_cache}/.", f"{drive_cache}/"], check=False)
print("LongCat inference verified; compiled kernels synced to Drive", flush=True)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = False   # send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        urllib.request.urlretrieve(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                   f"caddy_{tag.lstrip('v')}_linux_amd64.tar.gz", tgz)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            header_up X-Forwarded-Proto https
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                {AUTH_USER} {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import signal
import subprocess
import tarfile
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    import re
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        urllib.request.urlretrieve(base+asset,archive)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            return spawn('cloudflared',args)
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    import re
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- KEEP THIS CELL RUNNING
# The Colab session only lives while the `colab exec` connection is open and this cell blocks.
# The watchdog restarts any process from PROCS that died; a heartbeat line is printed every 10 min.
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=3600)
if AUTH == "caddy" and not CREDS_FROM_ENV:   # started from the Colab UI: show generated credentials
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

last_beat = time.time()
while True:
    time.sleep(30)
    for name, (proc, restart) in list(PROCS.items()):
        if proc.poll() is not None:
            print(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 15)}", flush=True)
            PROCS[name] = (restart(), restart)
    if time.time() - last_beat > 600:
        last_beat = time.time()
        print(f"[heartbeat] {time.strftime('%H:%M:%S')} " +
              " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items()), flush=True)
